In [ ]:
import requests
from collections import defaultdict
from itertools import islice

def chunked(iterable, size):
    it = iter(iterable)
    while True:
        chunk = list(islice(it, size))
        if not chunk:
            break
        yield chunk

def fetch_kegg_data(all_kos, chunk_size=100):
    session = requests.Session()

    ids_cache = {ko: {"pathway": [], "module": []} for ko in all_kos}

    # ---- Pathways ----
    for ko_chunk in chunked(all_kos, chunk_size):
        ko_str = "+".join(ko_chunk)
        url = f"https://rest.kegg.jp/link/pathway/{ko_str}"
        r = session.get(url)
        r.raise_for_status()

        for line in r.text.splitlines():
            ko, path = line.split("\t")
            if path.startswith("path:ko"):
                ids_cache[ko]["pathway"].append(path)

    # ---- Modules ----
    for ko_chunk in chunked(all_kos, chunk_size):
        ko_str = "+".join(ko_chunk)
        url = f"https://rest.kegg.jp/link/module/{ko_str}"
        r = session.get(url)
        r.raise_for_status()

        for line in r.text.splitlines():
            ko, mod = line.split("\t")
            if mod.startswith("md:"):
                ids_cache[ko]["module"].append(mod)

    return ids_cache

# Collect all unique KO IDs from input files
input_files = "/home/nanopore/projects/rna_seq_workflow/results/20250603_FCOM/functional_annotations/AOK_vs_MOK_combined_annotations.csv"  # List of input files
all_kos = set()

for file in input_files:
    df = pd.read_csv(file)
    df['KEGG_ko'] = df['KEGG_ko'].str.split(',')
    kos = df['KEGG_ko'].explode().str.strip().dropna()
    all_kos.update(kos)
    
# Fetch the pathway and module IDs for all unique KO IDs
ko_data = fetch_kegg_data(all_kos)